# AI Engineer 2주차 (신정호 강사님) - 내풀이 v1

정답지 미참조. 문제지 + `auto_grade.ipynb` 스펙 기준 풀이.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import json
import re
import time
from typing import List, Dict, Any, Tuple, Callable

## Q1. Tokenizer: BOS/EOS 및 Padding

In [ ]:
def preprocess_sentences(sentences: List[List[int]], max_l: int) -> Tuple[torch.Tensor, torch.Tensor]:
    batch_ids = []
    batch_masks = []
    for s in sentences:
        tokens = [1] + list(s) + [2]
        tokens = tokens[:max_l]
        mask = [1] * len(tokens)
        pad_len = max_l - len(tokens)
        tokens = tokens + [0] * pad_len
        mask = mask + [0] * pad_len
        batch_ids.append(tokens)
        batch_masks.append(mask)
    return torch.tensor(batch_ids, dtype=torch.long), torch.tensor(batch_masks, dtype=torch.long)

## Q2. Multi-Head Split

In [ ]:
def split_heads(tensor: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq, hidden = tensor.size()
    head_dim = hidden // num_heads
    return tensor.view(batch, seq, num_heads, head_dim).transpose(1, 2).contiguous()

## Q3. KV Cache Concatenation

In [ ]:
def append_to_kv_cache(prev_k: torch.Tensor, new_k: torch.Tensor) -> torch.Tensor:
    return torch.cat([prev_k, new_k], dim=-2)

## Q4. Temperature Softmax

In [ ]:
def get_probs_with_temp(logits: torch.Tensor, temp: float) -> torch.Tensor:
    return F.softmax(logits / temp, dim=-1)

## Q5. Tool Calling - JSON Schema

In [ ]:
currency_tool_schema = {
    "type": "object",
    "properties": {
        "amount": {"type": "number", "description": "환전할 금액"},
        "from_cur": {"type": "string", "description": "원본 통화 코드"},
        "to_cur": {"type": "string", "description": "대상 통화 코드", "default": "USD"}
    },
    "required": ["amount", "from_cur"]
}

## Q6. Argument Parser

In [ ]:
def parse_model_call(call_str: str) -> Dict[str, Any]:
    m = re.match(r"CALL:\s*(\w+)\((.*)\)\s*$", call_str.strip())
    if not m:
        return {}
    func_name = m.group(1)
    args_body = m.group(2).strip()
    args: Dict[str, Any] = {}
    if args_body:
        for pair in args_body.split(","):
            if "=" not in pair:
                continue
            k, v = pair.split("=", 1)
            args[k.strip()] = v.strip()
    return {"func_name": func_name, "args": args}

## Q7. MHA KV Cache Update

In [ ]:
def update_mha_kv_cache(prev_k: torch.Tensor, new_k_proj: torch.Tensor, num_heads: int) -> torch.Tensor:
    batch, seq_new, hidden = new_k_proj.size()
    head_dim = hidden // num_heads
    new_k = new_k_proj.view(batch, seq_new, num_heads, head_dim).transpose(1, 2)
    return torch.cat([prev_k, new_k], dim=-2)

## Q8. Multi-Step Tool Chaining (킬러)

In [ ]:
def multi_step_tool_agent(item: str, tools: Dict[str, Callable]) -> str:
    pid = tools["find_product_id"](item)
    if pid is None:
        return "ERROR: PRODUCT_NOT_FOUND"
    rate = tools["get_discount_rate"](pid)
    if rate > 0.5:
        return "ERROR: INVALID_DISCOUNT"
    price = tools["calculate_price"](pid, rate)
    return f"Final Price for {item} is {price}"

## Q9. Sliding Window Batch KV Cache (킬러)

In [ ]:
def sliding_window_batch_kv(cache: torch.Tensor, new_val: torch.Tensor, window_size: int) -> torch.Tensor:
    combined = torch.cat([cache, new_val], dim=-2)
    return combined[..., -window_size:, :]

## Q10. LLM API Safety & Retry (킬러)

In [ ]:
def call_llm_api_safely(prompt: str, max_tokens: int, api_fn: Callable) -> Dict[str, Any]:
    if len(prompt.split()) > max_tokens:
        return "ERROR: CONTEXT_EXCEEDED"

    max_attempts = 3
    for attempt in range(1, max_attempts + 1):
        code, resp = api_fn(prompt)
        if code == 200:
            return {"status": "success", "response": resp, "attempts": attempt}
        if code == 429:
            if attempt < max_attempts:
                time.sleep(1)
                continue
            return {"status": "fail", "reason": "RATE_LIMIT_EXCEEDED"}
        return {"status": "fail", "reason": "SERVER_ERROR"}

    return {"status": "fail", "reason": "RATE_LIMIT_EXCEEDED"}

## 자체 검증

In [ ]:
# Q1
ids, masks = preprocess_sentences([[10, 20], [30]], 5)
assert ids[0].tolist() == [1, 10, 20, 2, 0]
assert masks[1].tolist() == [1, 1, 1, 0, 0]
assert ids.dtype == torch.long
print("Q1 PASS")

# Q2
t = torch.randn(2, 4, 16)
r2 = split_heads(t, num_heads=4)
assert r2.shape == (2, 4, 4, 4)
assert torch.allclose(r2[0, 0, 0, :], t[0, 0, :4])
print("Q2 PASS")

# Q3
r3 = append_to_kv_cache(torch.zeros(1, 1, 4, 16), torch.ones(1, 1, 1, 16))
assert r3.shape == (1, 1, 5, 16) and r3[0, 0, 4, 0].item() == 1.0
print("Q3 PASS")

# Q4
logits = torch.tensor([1.0, 2.0, 10.0])
p_low = get_probs_with_temp(logits, 0.1)
p_high = get_probs_with_temp(logits, 10.0)
assert torch.isclose(p_low.sum(), torch.tensor(1.0))
assert p_low[2] > p_high[2]
print("Q4 PASS")

# Q5
assert "amount" in currency_tool_schema["required"]
assert currency_tool_schema["properties"]["to_cur"]["default"] == "USD"
print("Q5 PASS")

# Q6
r6 = parse_model_call("CALL: order(item=GPU, qty=2)")
assert r6["func_name"] == "order" and r6["args"]["item"] == "GPU"
print("Q6 PASS")

# Q7
r7 = update_mha_kv_cache(torch.zeros(1, 2, 2, 8), torch.ones(1, 1, 16), 2)
assert r7.shape == (1, 2, 3, 8) and torch.all(r7[:, :, 2, :] == 1.0)
print("Q7 PASS")

# Q8
tools = {
    "find_product_id": lambda n: "p1" if n == "Phone" else None,
    "get_discount_rate": lambda pid: 0.2 if pid == "p1" else 0.6,
    "calculate_price": lambda pid, r: int(100 * (1 - r)),
}
assert multi_step_tool_agent("Phone", tools) == "Final Price for Phone is 80"
assert multi_step_tool_agent("Desk", tools) == "ERROR: PRODUCT_NOT_FOUND"
print("Q8 PASS")

# Q9
cache = torch.arange(16).view(1, 1, 4, 4).float()
new_v = torch.full((1, 1, 1, 4), 99.0)
r9 = sliding_window_batch_kv(cache, new_v, 4)
assert r9.shape == (1, 1, 4, 4) and r9[0, 0, 0, 0].item() == 4.0
print("Q9 PASS")

# Q10
assert call_llm_api_safely("a b c d e", 2, lambda p: (429, "x")) == "ERROR: CONTEXT_EXCEEDED"
r10_a = call_llm_api_safely("hi", 10, lambda p: (429, "L"))
assert r10_a["reason"] == "RATE_LIMIT_EXCEEDED"
r10_b = call_llm_api_safely("hi", 10, lambda p: (500, "E"))
assert r10_b["reason"] == "SERVER_ERROR"
r10_c = call_llm_api_safely("hi", 10, lambda p: (200, "OK"))
assert r10_c == {"status": "success", "response": "OK", "attempts": 1}
print("Q10 PASS")

print("\n=== ALL PASS ===")